In [59]:
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"
v = embed.encode(q)

ModuleNotFoundError: No module named 'onnxruntime'

In [ ]:
v[0]

np.float64(-0.020582036807885073)

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [ ]:
d = "02-vector-search/lessons/07-sqlitesearch-vector.md"
dv = embed.encode(d)

In [ ]:
v.dot(dv)

np.float64(0.3770526250449936)

In [ ]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
texts = [chunk["content"] for chunk in chunks]

from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)
X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [ ]:
query = "What metric do we use to evaluate a search engine?"
v_query = embed.encode(query)

In [ ]:
results = vindex.search(v_query, num_results=5)
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [ ]:
vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(X, chunks)

In [ ]:
from minsearch import Index

text_index = Index(text_fields=["content"])
text_index.fit(chunks)

In [ ]:
query = "How do I store vectors in PostgreSQL?"
# vector search (reuse what you built in Q4)
v_query = embed.encode(query)
vector_results = vindex.search(v_query, num_results=5)

# text search
text_results = text_index.search(query, num_results=5)

In [ ]:
vector_filenames = [r["filename"] for r in vector_results]
text_filenames = [r["filename"] for r in text_results]

print("Vector:", vector_filenames)
print("Text:", text_filenames)

Vector: ['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']
Text: ['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']


In [ ]:
set(vector_filenames) - set(text_filenames)

{'02-vector-search/lessons/08-pgvector.md'}

In [ ]:
query = "How do I give the model access to tools?"

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
v_query = embed.encode(query)
vector_results = vindex.search(v_query, num_results=5)
text_results = text_index.search(query, num_results=5)

results = rrf([vector_results, text_results])

In [ ]:
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'